# Spark Tune - Databricks ML Pipeline Demo

End-to-end ML pipeline reading data from a Databricks catalog and demonstrating:

5. **pre-processing** - Data preprocessing for model training and insight generation

In [0]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore')

---
## 1. Load Data from Databricks Catalog

In [0]:
# CATALOG_NAME = "aidetic_databricks"
# SCHEMA_NAME = "default"
# FEATURE_SCHEMA_NAME = "feature_store"

# # Table Names
# credit_card_transactions_table_name = f"{CATALOG_NAME}.{SCHEMA_NAME}.credit_card_transactions"

# ftool_feature_table_name = f"{CATALOG_NAME}.{FEATURE_SCHEMA_NAME}.hdfc_demo_credit_card_trans_featuretools"
# tsfresh_feature_table_name = f"{CATALOG_NAME}.{FEATURE_SCHEMA_NAME}.hdfc_demo_credit_card_trans_tsfresh"
# auto_feature_table_name = f"{CATALOG_NAME}.{FEATURE_SCHEMA_NAME}.hdfc_demo_credit_card_trans_auto"



# # Reading Original Data
# df = spark.read.table(credit_card_transactions_table_name)

# print(f"Dataset shape: {df.count():,} rows x {len(df.columns)} columns")
# df.printSchema()

CATALOG_NAME = "aidetic_databricks"
SCHEMA_NAME = "default"
TABLE_NAME = "hdfc_demo_bank_customers"

FEATURE_SCHEMA_NAME = "feature_store"
FEATURE_TABLE_NAME = "hdfc_demo_customer_transaction_featuretools"
TSFRESH_FEATURE_TABLE_NAME = "hdfc_demo_cust_trans_tsfresh"
AUTO_FEATURE_TABLE_NAME = f"{TABLE_NAME}_auto"


# # Table Names
# credit_card_transactions_table_name = f"{CATALOG_NAME}.{SCHEMA_NAME}.credit_card_transactions"
credit_card_transactions_table_name = f"{CATALOG_NAME}.{SCHEMA_NAME}.{TABLE_NAME}"

# ftool_feature_table_name = f"{CATALOG_NAME}.{FEATURE_SCHEMA_NAME}.hdfc_demo_credit_card_trans_featuretools"
# tsfresh_feature_table_name = f"{CATALOG_NAME}.{FEATURE_SCHEMA_NAME}.hdfc_demo_credit_card_trans_tsfresh"
# auto_feature_table_name = f"{CATALOG_NAME}.{FEATURE_SCHEMA_NAME}.hdfc_demo_credit_card_trans_auto"
ftool_feature_table_name = f"{CATALOG_NAME}.{FEATURE_SCHEMA_NAME}.{FEATURE_TABLE_NAME}"
tsfresh_feature_table_name = f"{CATALOG_NAME}.{FEATURE_SCHEMA_NAME}.{TSFRESH_FEATURE_TABLE_NAME}"
auto_feature_table_name = f"{CATALOG_NAME}.{FEATURE_SCHEMA_NAME}.{AUTO_FEATURE_TABLE_NAME}"





# Reading Original Data
df = spark.read.table(credit_card_transactions_table_name)

print(f"Dataset shape: {df.count():,} rows x {len(df.columns)} columns")
df.printSchema()

In [0]:
# from databricks.feature_engineering import FeatureEngineeringClient, FeatureLookup

# fe = FeatureEngineeringClient()

# feature_lookups = [
#     FeatureLookup(
#         table_name=ftool_feature_table_name,
#         lookup_key='trans_num',
#         timestamp_lookup_key='trans_date_trans_time',
#     ),
#     FeatureLookup(
#         table_name=tsfresh_feature_table_name,
#         lookup_key='trans_num',
#         timestamp_lookup_key='trans_date_trans_time',
#     ),
#     FeatureLookup(
#         table_name=auto_feature_table_name,
#         lookup_key='trans_num',
#         timestamp_lookup_key='trans_date_trans_time',
#     ),                  
# ]

from databricks.feature_engineering import FeatureEngineeringClient, FeatureLookup

fe = FeatureEngineeringClient()

# Get feature columns for each table (excluding lookup keys and contact_timestamp)
ftool_df = spark.read.table(ftool_feature_table_name)
ftool_features = [col for col in ftool_df.columns if col not in ['customer_id', 'contact_timestamp']]

tsfresh_df = spark.read.table(tsfresh_feature_table_name)
tsfresh_features = [col for col in tsfresh_df.columns if col not in ['customer_id']]

auto_df = spark.read.table(auto_feature_table_name)
auto_features = [col for col in auto_df.columns if col not in ['customer_id', 'contact_timestamp']]

feature_lookups = [
    FeatureLookup(
        table_name=ftool_feature_table_name,
        lookup_key='customer_id',
        feature_names=ftool_features,
    ),
    FeatureLookup(
        table_name=tsfresh_feature_table_name,
        lookup_key='customer_id',
        feature_names=tsfresh_features,
    ),
    FeatureLookup(
        table_name=auto_feature_table_name,
        lookup_key='customer_id',
        timestamp_lookup_key='contact_timestamp',
        feature_names=auto_features,
    ),
                   
]

In [0]:

# Since the timestamp columns would likely cause the model to overfit the data
# unless additional feature engineering was performed, exclude them to avoid training on them.
exclude_columns = ["trans_num", "trans_date_trans_time"]

# Create the training set that includes the raw input data merged with corresponding features from both feature tables
training_set = fe.create_training_set(
    df=df,
    feature_lookups=feature_lookups,
    label="responded",
    exclude_columns=exclude_columns,
)

# Load the TrainingSet into a dataframe which can be passed into sklearn for training a model
training_df = training_set.load_df()

---
## 10. Insight Analyzer - Microsegments & Lift vs Support

SparkBeyond-style feature discovery that identifies:
- **Lift**: How much better a feature condition performs vs. baseline (e.g., x3.09 = 3x better)
- **Support**: Percentage of data covered by the condition
- **RIG**: Relative Information Gain - information value about the target
- **Microsegments**: Powerful combinations of feature conditions

In [0]:
from backend.core.discovery import Problem, SchemaChecks

problem = Problem(
    target="responded",
    type="classification",
    desired_result=1,
    # date_column="trans_date_trans_time"
    date_column="contact_timestamp"
)

schema_checker = SchemaChecks(dataframe=df, problem=problem)
schema_info = schema_checker.check()

print(f"Problem Type: {problem.type}")
print(f"Target Column: {problem.target}")
print(f"Desired Result: {problem.desired_result}")
print(f"\nSchema Summary:")
print(f"  Categorical columns: {len(schema_info['categorical'])}")
print(f"  Numerical columns: {len(schema_info['numerical'])}")
print(f"  Boolean columns: {len(schema_info['boolean'])}")

In [0]:
from backend.core.features.insight_analyzer import FeatureInsightAnalyzer

print("FEATURE INSIGHT ANALYSIS")
print("=" * 60)

insight_analyzer = FeatureInsightAnalyzer(
    df=training_df,      # use original data (not transformed)
    problem=problem,
    schema_checks=schema_checker,
    n_bins=10,                 # bins for numeric features
    min_support=0.01,          # minimum 1% support
    min_lift=1.1               # minimum 10% lift over baseline
)

result = insight_analyzer.get_analysis_result(discover_microsegments=True)

print(f"\nAnalysis Summary:")
print(f"  Target Class: {result.target_class}")
print(f"  Baseline Rate: {result.baseline_rate*100:.2f}%")
print(f"  Total Records: {result.total_count:,}")
print(f"  Total Insights Found: {result.summary['total_insights']}")
print(f"  Microsegments Found: {result.summary['total_microsegments']}")

In [0]:
# Top insights sorted by lift
print("TOP 20 FEATURE INSIGHTS (Sorted by Lift):")
print("-" * 70)

insights_df = insight_analyzer.to_dataframe(top_n=20)
display_cols = ['Condition', 'Lift', 'Support', 'Support_Count', 'RIG', 'Class_Rate']
print(insights_df[display_cols].to_string(index=False))

In [0]:
# Display microsegments (feature combinations)
print("TOP MICROSEGMENTS (Feature Combinations):")
print("-" * 70)

if result.microsegments:
    for i, micro in enumerate(result.microsegments[:10], 1):
        print(f"\n{i}. {micro.name}")
        print(f"   Lift: x{micro.lift:.2f} | Support: {micro.support*100:.1f}% ({micro.support_count:,}) | RIG: {micro.rig:.3f}")
        print(f"   Class Rate: {micro.class_rate*100:.1f}% vs Baseline: {micro.baseline_rate*100:.1f}%")
else:
    print("No microsegments found that improve over individual features.")

In [0]:
from IPython.display import Image# Lift vs Support scatter plot (SparkBeyond-style)

print("Generating Lift vs Support Scatter Plot...")
fig = insight_analyzer.plot_lift_support_scatter(
    top_n=50,
    highlight_microsegments=True,
    save_path='insight_lift_support.png'
)

display(Image(filename='insight_lift_support.png'))

In [0]:
insight_analyzer._microsegments

In [0]:
from IPython.display import Image# Top insights bar chart

print("Generating Top Insights Bar Chart...")
fig = insight_analyzer.plot_top_insights(
    top_n=15,
    metric='lift',
    save_path='insight_top_features.png'
)

display(Image(filename='insight_top_features.png'))

In [0]:
# Full insights table for exploration
print("FULL INSIGHTS TABLE:")
display(insight_analyzer.display_insights_table(top_n=50))

---
## Summary

### Cumulative Feature Pipeline

```
df (raw) ──► featuretools DFS ──► tsfresh time-series ──► AutoFeatureGenerator ──► Spark ML Pipeline ──► XGBoost / SHAP / Insights
              (merge back)         (merge back)            (interactions, bins)     (encode + vectorize)
```

| Step | Library | What it adds |
|------|---------|--------------|
| 3 | **ydata-profiling** | Data exploration, alerts & recommendations |
| 5 | **featuretools** | DFS transform features (add, subtract, multiply numeric) |
| 6 | **tsfresh** | Entity-level time-series statistics (mean, std, min, max, ...) |
| 7 | **AutoFeatureGenerator** | Interactions, binning, datetime extraction on enriched df |
| 8 | **XGBoost** | SparkXGBoost training with feature importance |
| 9 | **SHAP** | Shapley value explanations (summary, bar, waterfall) |
| 10 | **Insight Analyzer** | Lift, Support, RIG analysis & microsegment discovery |

### Key Metrics from Insight Analysis:
- **Lift**: How much better a feature condition performs vs. baseline
- **Support**: Percentage of data covered by the condition
- **RIG**: Relative Information Gain - how much information the feature provides about the target

In [0]:
# Cleanup
insight_analyzer.cleanup()
print("Demo complete!")